# Solutions — Testing

One solution per exercise and per mini challenge, in lesson order.
Read these **after** you have tried. A solution you have not attempted teaches nothing.

The Vitest and Testing Library answers are given as the code to put in `react-scratch`; every one
of them was installed and run while writing these lessons, on `vitest 5.0.0`,
`@testing-library/react 16.3.3`, `@testing-library/user-event 14.6.7`,
`@testing-library/jest-dom 7.0.1` and `jsdom 30.0.1`.

### LESSON 84 — Exercise

In [ ]:
// L84 solution — a slightly bigger runner

const l84sLog = [];
let l84sGroup = null;

function l84sDescribe(name, body) {
  l84sGroup = name;
  l84sLog.push(`${name}`);
  body();
  l84sGroup = null;
}

function l84sIt(name, body) {
  const indent = l84sGroup ? "    " : "  ";
  try {
    body();
    l84sLog.push(`${indent}PASS  ${name}`);
  } catch (error) {
    l84sLog.push(`${indent}FAIL  ${name}\n${indent}      ${error.message}`);
  }
}

function l84sExpect(actual) {
  const fail = (message) => { throw new Error(message); };
  return {
    toBe: (e) => Object.is(actual, e) || fail(`expected ${JSON.stringify(e)}, got ${JSON.stringify(actual)}`),
    toEqual: (e) => JSON.stringify(actual) === JSON.stringify(e) || fail(`expected ${JSON.stringify(e)}, got ${JSON.stringify(actual)}`),
    toContain: (e) =>
      (typeof actual === "string" ? actual.includes(e) : Array.isArray(actual) && actual.includes(e)) ||
      fail(`expected ${JSON.stringify(actual)} to contain ${JSON.stringify(e)}`),
    toHaveLength: (n) =>
      actual?.length === n || fail(`expected length ${n}, got ${actual?.length}`),
  };
}

function l84sValidate(values) {
  const errors = {};
  if (!values.name?.trim()) errors.name = "Name is required";
  if (!values.email?.includes("@")) errors.email = "That doesn't look like an email";
  return errors;
}

l84sDescribe("validate", () => {
  l84sIt("requires a name", () =>
    l84sExpect(l84sValidate({ name: "", email: "a@b" })).toEqual({ name: "Name is required" }));
  l84sIt("accepts a complete form", () =>
    l84sExpect(Object.keys(l84sValidate({ name: "Ada", email: "ada@example.com" }))).toHaveLength(0));
  l84sIt("mentions the field in the message", () =>
    l84sExpect(l84sValidate({ name: "", email: "a@b" }).name).toContain("Name"));
});

for (const line of l84sLog) console.log(line);

**2. The setup working.** With the `test` block in `vite.config.js` and no test files yet,
`npx vitest run` reports **no test files found**. That is the goal of the step: it proves Vitest
started, read your config and looked for tests, which separates "the setup is wrong" from "my test
is wrong" before you have written a line.

**3. What the real runner tells you that yours does not.** A failing Vitest assertion prints the
file and line, the source of the failing line with a caret under it, a diff of expected against
received, and — for a Testing Library query — the rendered DOM. Yours prints a sentence. That
difference is most of what a test runner is for: not deciding pass or fail, but making a failure
readable in ten seconds.

### LESSON 84 — Mini challenge

In [ ]:
// L84 solution — sorting the candidate tests

const l84sCandidates = [
  ["reducer handles FILTER_CHANGED", "worth it", "pure logic with branches — cheapest test there is"],
  ["component calls useState twice", "not worth it", "an implementation detail; a refactor breaks it while the app works"],
  ["searching 'ada' shows only Ada's row", "worth it", "the behaviour the feature exists for"],
  ["spinner has the class spinner-lg", "not worth it", "markup, not behaviour — fails on a redesign that broke nothing"],
  ["empty state appears for []", "worth it", "a branch nobody clicks through by hand"],
  ["service sends the right URL", "worth it", "pure logic, and wrong URLs fail silently in the UI"],
  ["heading text is 'Employee Directory'", "not worth it", "you would not ship that broken; it fails on a copy edit"],
  ["clicking a row opens the detail panel", "worth it", "behaviour, and it spans two components"],
  ["Router navigates to /employees/3", "not worth it", "that is React Router's own test; test what YOUR page shows at that URL"],
  ["fixed bug: department no longer clears the search", "worth it", "a regression test for a bug that really happened"],
];

function l84sVerdict(item) {
  const row = l84sCandidates.find(([name]) => name === item);
  return row ? { verdict: row[1], why: row[2] } : null;
}

for (const [name] of l84sCandidates) {
  const { verdict, why } = l84sVerdict(name);
  console.log(`${verdict.padEnd(14)} ${name}\n               ${why}`);
}

// The sentence that rules out 2 and 4: test what the component DOES, not how it does it. Both
// assert something invisible to the user — a Hook call count and a class name — so both fail on a
// refactor that changes nothing anyone can see, which is the definition of a test that costs more
// than it gives.
//
// Why 9 is on the list even though it looks like behaviour: "Router navigates to /employees/3" is
// a claim about React Router, and React Router is tested by its authors. The behaviour that is
// yours is what the user sees after that navigation — the detail page for employee 3 — and testing
// THAT covers your route config, your component and your data lookup at once. Test your app at the
// boundary of your own code.

### LESSON 85 — Exercise

In [ ]:
// L85 solution — five cases for a validator, including the two that crash it

// the runner again, one binding set per notebook
const l85sLog = [];
function l85sIt(name, body) {
  try { body(); l85sLog.push(`  PASS  ${name}`); }
  catch (error) { l85sLog.push(`  FAIL  ${name}\n        ${error.message}`); }
}
function l85sExpect(actual) {
  const fail = (m) => { throw new Error(m); };
  return {
    toEqual: (e) => JSON.stringify(actual) === JSON.stringify(e) || fail(`expected ${JSON.stringify(e)}, got ${JSON.stringify(actual)}`),
    toHaveProperty: (k) => (actual && k in actual) || fail(`expected a "${k}" key, got ${JSON.stringify(actual)}`),
  };
}

// the FIXED validator: optional chaining and a String() so it survives undefined and non-strings
function l85sValidate(values = {}) {
  const errors = {};
  const name = values.name;
  const email = values.email;
  if (String(name ?? "").trim() === "") errors.name = "Name is required";
  if (!String(email ?? "").includes("@")) errors.email = "That doesn't look like an email";
  return errors;
}

l85sIt("a valid form has no errors", () =>
  l85sExpect(l85sValidate({ name: "Ada", email: "ada@example.com" })).toEqual({}));

l85sIt("one rule failing", () =>
  l85sExpect(l85sValidate({ name: "", email: "ada@example.com" })).toEqual({ name: "Name is required" }));

l85sIt("two rules failing on different fields", () =>
  l85sExpect(l85sValidate({ name: "  ", email: "ada" })).toEqual({
    name: "Name is required",
    email: "That doesn't look like an email",
  }));

l85sIt("an empty object reports both fields", () => {
  const errors = l85sValidate({});
  l85sExpect(errors).toHaveProperty("name");
  l85sExpect(errors).toHaveProperty("email");
});

l85sIt("undefined values do not crash it", () =>
  l85sExpect(l85sValidate({ name: undefined, email: undefined })).toEqual({
    name: "Name is required",
    email: "That doesn't look like an email",
  }));

for (const line of l85sLog) console.log(line);

// The point of the last two: LESSON 79's validator used `value.trim()` and `value.includes("@")`,
// which throw on undefined. The test does not "find a bug in the test" — it finds that the
// function has an input it cannot survive, and in a real form that input arrives the first time
// someone adds a field to the values object and forgets to initialise it.

**2 and 3 — in `react-scratch`.** The reducer test file, and the two lines that matter:

```js
import { describe, expect, it } from "vitest";
import { reducer } from "./reducer.js";

const state = { items: [{ id: 1, text: "write tests", done: false }] };

describe("reducer", () => {
  it("adds an item", () => {
    expect(reducer(state, { type: "added", id: 2, text: "run them" }).items).toHaveLength(2);
  });

  it("does not mutate the state it was given", () => {
    reducer(state, { type: "added", id: 2, text: "run them" });
    expect(state.items).toHaveLength(1);          // ← the assertion that catches the real bug
  });

  it("clears nothing when nothing is done", () => {
    expect(reducer(state, { type: "cleared" }).items).toHaveLength(1);
  });

  it("throws on an unknown action", () => {
    expect(() => reducer(state, { type: "nope" })).toThrow("Unknown action");
  });
});
```

Swap the spread for `state.items.push(...)` and **only the second test fails** — the first still
passes, because the pushed item is in the array either way. That is exactly why the no-mutation
test is worth writing: every other test in the file is blind to the bug.

**Common mistake:** writing the no-mutation test as
`expect(reducer(state, action)).not.toEqual(state)`. That compares the *result* with the input and
says nothing about whether the input was modified. The assertion has to be about `state` itself,
after the call.

### LESSON 85 — Mini challenge

In [ ]:
// L85 solution — tests that cannot fail

const l85sTautologies = [
  ["expect(next).toBeDefined()", "cannot fail", "any object is defined; a reducer returning {} passes"],
  ["expect(typeof validate({})).toBe('object')", "cannot fail", "a validator that always returns {} passes"],
  ["expect(next).toEqual(next)", "cannot fail", "compares a value with itself — always true"],
  ["expect(visible.length).toBeGreaterThanOrEqual(0)", "cannot fail", "no array has a negative length"],
];

function l85sCanFail(test) {
  const row = l85sTautologies.find(([name]) => name === test);
  return row ? { verdict: row[1], why: row[2] } : { verdict: "unknown", why: "" };
}

for (const [name] of l85sTautologies) {
  const { verdict, why } = l85sCanFail(name);
  console.log(`${verdict.padEnd(12)} ${name}\n             ${why}`);
}

// --- the rewrites, each run against a BROKEN implementation to prove it fails ---
function l85sBrokenReducer() { return {}; }                  // returns nothing useful
function l85sBrokenValidate() { return {}; }                 // never reports an error
function l85sMutatingReducer(state, action) {                // mutates
  state.items.push({ id: action.id });
  return state;
}
function l85sBrokenFilter() { return []; }                   // filters everything away

const l85sState = { items: [{ id: 1 }] };
const l85sChecks = [
  ["adds an item", () => l85sBrokenReducer(l85sState, { type: "added", id: 2 }).items?.length === 2],
  ["reports a missing name", () => l85sBrokenValidate({ name: "" }).name === "Name is required"],
  ["does not mutate", () => {
    const before = l85sState.items.length;
    l85sMutatingReducer({ items: [...l85sState.items] }, { type: "added", id: 2 });
    const probe = { items: [{ id: 1 }] };
    l85sMutatingReducer(probe, { type: "added", id: 2 });
    return probe.items.length === before;
  }],
  ["filters the list", () => l85sBrokenFilter([{ name: "Ada" }, { name: "Bo" }], "ad").length === 1],
];

console.log("\nthe rewritten assertions, against broken code:");
for (const [name, check] of l85sChecks) {
  console.log(`  ${check() ? "PASS — still not failing!" : "FAIL — good, it can fail"}  ${name}`);
}

// The shape all four share: the assertion is about the TYPE or the EXISTENCE of the result, not
// about its VALUE. "It returned something" is true of every implementation, including an empty
// one, so the test passes for reasons that have nothing to do with correctness.
//
// The question to ask of every assertion: "what implementation would make this fail?" If you
// cannot name one — or if the only one is a crash — the assertion is decoration. Writing the
// broken version first, as above, is the fastest way to be sure.

### LESSON 86 — Exercise

**1. Deleting `role="status"`.** The `findByRole("status")` query fails after its timeout, and
Testing Library prints the rendered DOM plus the list of accessible roles it *did* find. Real
output from that failure, with the role removed:

```text
TestingLibraryElementError: Unable to find an accessible element with the role "status"

Here are the accessible roles:

  paragraph:
  Name "":
  <p />

  button:
  Name "increment":
  <button />
```

It suggests exactly what is available — here, that the message is now a plain `paragraph` with no
role, so either the test should use `getByText("that is plenty")` or, better, the component should
keep the `role="status"` so screen readers announce it. The test noticed an accessibility
regression, which is the argument for role queries in one paragraph.

**2. A deliberately failing assertion.** The output prints the whole rendered DOM:

```text
Unable to find an element with the text: Count: 5. This could be because the text is broken up by
multiple elements…

<body>
  <div>
    <div>
      <p>Count: 0</p>
      <button>increment</button>
    </div>
  </div>
</body>
```

Why that makes `getByTestId` mostly unnecessary: the usual reason people reach for a test id is
"I cannot find the element". The failure output shows you the entire document and every accessible
role in it, so you can *see* what to query. A test id is then a choice to make the element findable
only to tests — which is exactly the element a user would also struggle to find.

**3. The controlled input, and the label.**

```jsx
it("types into the name field", async () => {
  const user = userEvent.setup();
  render(<NameForm />);

  const input = screen.getByLabelText("Name");
  await user.type(input, "Ada");

  expect(input).toHaveValue("Ada");
});
```

Break the association — remove the `htmlFor`/`id` pair, or the wrapping `<label>` — and
`getByLabelText` fails. What that tells you is **about the component**: an input with no
associated label is an input a screen reader announces as unlabelled, and one that does not focus
when the user clicks its text. The test failure is a real defect report, not a testing
inconvenience. (LESSON 9 taught `htmlFor` for exactly this.)

**4. Testing a Mini-project 3 component.** The usual answer is yes, something had to change — most
often a component that fetched its own data had to accept the data, or a loader, as a prop. That
is an improvement independent of testing: the component becomes usable with any data source, its
inputs are visible in its signature, and it stops being coupled to one service module. LESSON 66's
"separate data from markup" arrives here as a consequence of trying to test the thing.

### LESSON 86 — Mini challenge

The component, with `load` as a prop so the test can supply it:

```jsx
import { useEffect, useState } from "react";

export default function Employees({ load }) {
  const [status, setStatus] = useState("loading");
  const [rows, setRows] = useState([]);

  useEffect(() => {
    let ignore = false;
    setStatus("loading");
    load()
      .then((data) => {
        if (ignore) return;
        setRows(data);
        setStatus(data.length === 0 ? "empty" : "success");
      })
      .catch(() => { if (!ignore) setStatus("error"); });
    return () => { ignore = true; };
  }, [load]);

  if (status === "loading") return <p role="status">Loading employees…</p>;
  if (status === "error") return <p role="alert">Couldn't load employees. Try again.</p>;
  if (status === "empty") return <p>No employees yet.</p>;

  return <ul>{rows.map((row) => <li key={row.id}>{row.name}</li>)}</ul>;
}
```

and the four tests, which pass as written (`Test Files 1 passed · Tests 4 passed`):

```jsx
import { render, screen } from "@testing-library/react";
import { expect, it } from "vitest";
import Employees from "./Employees.jsx";

const later = (value, ms = 30) => () => new Promise((r) => setTimeout(() => r(value), ms));
const failing = () => () => Promise.reject(new Error("boom"));

it("shows a loading state first", () => {
  render(<Employees load={later([{ id: 1, name: "Ada" }])} />);
  expect(screen.getByRole("status")).toHaveTextContent("Loading employees…");
});

it("shows an error when the load fails", async () => {
  render(<Employees load={failing()} />);
  expect(await screen.findByRole("alert")).toHaveTextContent("Couldn't load employees.");
});

it("shows an empty state for no rows", async () => {
  render(<Employees load={later([])} />);
  expect(await screen.findByText("No employees yet.")).toBeInTheDocument();
});

it("lists the rows on success", async () => {
  render(<Employees load={later([{ id: 1, name: "Ada" }, { id: 2, name: "Grace" }])} />);
  const items = await screen.findAllByRole("listitem");
  expect(items).toHaveLength(2);
  expect(items[0]).toHaveTextContent("Ada");
});
```

**1. The one most people get wrong first** is the loading test — written with `findByRole` instead
of `getByRole`. `findBy` waits, and by the time it resolves the promise has settled and the loading
state is gone, so it fails with the *success* DOM printed underneath. Loading is the state that is
there **synchronously**, so it is the one query in the file that must be `getBy`. The three async
states need `findBy`; using `getBy` for those fails immediately with the loading DOM printed, which
is the same lesson from the other side.

**2. Where `load` as a prop comes from.** LESSON 66 — separate data-fetching from markup — and
LESSON 67, where a component's props are its interface. Because the loader arrives as a prop, the
test hands it a function and there is no network, no `vi.mock`, and no module interception to
learn. Had the component imported the service directly, you would have had to mock the module,
which couples the test to the file layout: move the service and the test breaks while the app
works, which is precisely the kind of test LESSON 84 said not to write.

**3. Which test finds a real bug.** Almost always the **empty** one. `[]` is falsy-adjacent in the
worst way: the common bug is `data.length === 0` never being checked, so the component renders an
empty `<ul>` and the user sees a blank area with no explanation. Nobody notices by hand because
the developer's test data always has rows — you have to go out of your way to produce an empty
result, and the test does it in one line.

> **A note on what you have installed.** Vitest, jsdom and the three Testing Library packages live
> in `react-scratch` only. The shared playground is still React and Vite with no test tooling, and
> Mini-project 4 will install its own — deliberately, so that each project's dependencies are a
> decision you made rather than one you inherited.